# 02 — Exploratory data analysis
**Level: beginner → intermediate.** Study distributions and feature/data quality before fitting any model. Charts below are descriptive, not proof of attacks.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from nohack.datasets.prepare import validate_canonical_table
p = Path(os.environ.get("NOHACK_DATASET", "data/interim/labeled_nohackflow.csv"))
if not p.is_file():
    raise FileNotFoundError("Set NOHACK_DATASET to your verified canonical CSV/Parquet path")
df = pd.read_parquet(p) if p.suffix == ".parquet" else pd.read_csv(p)
df = validate_canonical_table(df)
df["is_attack"] = df["label"].str.upper().ne("BENIGN")
print(df.groupby(["label"]).size().sort_values(ascending=False).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df["label"].value_counts().head(12).plot(kind="bar", ax=ax)
ax.set(xlabel="Verified label", ylabel="Flow count", title="Class representation")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
cols = ["duration_ms", "total_packet_count", "total_byte_count", "packets_per_second"]
print(df.groupby("is_attack")[cols].agg(["median", "mean"]).round(2).to_string())
fig, ax = plt.subplots(figsize=(8, 4))
df[cols].clip(lower=0).apply(lambda s: s.map(lambda v: __import__("math").log1p(v))).boxplot(ax=ax)
ax.set_title("log1p flow feature ranges (do not confuse scale with importance)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout(); plt.show()

In [ ]:
# Check whether independent groups have diverse labels.
by_group = pd.crosstab(df.group_id, df.is_attack)
print(by_group.describe().to_string())
print("Number of groups with both classes:", int((by_group.gt(0).all(axis=1)).sum()))

### Questions to answer
Which labels are rare? Which groups contain only one class? Are there implausible durations, impossible rates or duplicated flow vectors? Document anomalies and inspect provenance before any deletion.